# Notebook 04 — PCA + K-Means + DBSCAN

**Input:** `data/processed/features.csv` (42 features, ~42,975 rows)

**Goal:** Reduce dimensionality with PCA, then apply two clustering-based anomaly detectors.

> Label isolation: `is_anomaly` loaded here but stored separately — NEVER passed to any model.


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

sys.path.insert(0, os.path.join('..', 'src'))
from detection_models import KMeansAnomalyDetector, DBSCANAnomalyDetector

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Load features
df = pd.read_csv('../data/processed/features.csv', parse_dates=['timestamp'])

ID_COLS    = ['timestamp', 'server_id', 'service_type']
LABEL_COLS = ['is_anomaly', 'anomaly_type']
FEAT_COLS  = [c for c in df.columns if c not in ID_COLS + LABEL_COLS]

X         = df[FEAT_COLS].values          # feature matrix (never contains labels)
labels_df = df[ID_COLS + LABEL_COLS].copy()  # kept separate

# Assertion: verify label isolation
assert all(c not in FEAT_COLS for c in LABEL_COLS), 'LABEL LEAK!'
print(f'Feature matrix X: {X.shape}  |  Labels stored separately: {labels_df.shape}')


## 1. PCA — Scree Plot and Component Selection

**WHY PCA before clustering:**  
K-Means and DBSCAN rely on distance metrics. In 42 dimensions, all points tend to be
equidistant from each other (curse of dimensionality), making clustering unreliable.
PCA compresses the correlated features (latency ↔ response_time, etc.) into a smaller
set of uncorrelated components, restoring the discriminative power of distance metrics.


In [ ]:
# Fit full PCA to inspect explained variance
pca_full = PCA(random_state=42)
pca_full.fit(X)

cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
n_85   = np.argmax(cumvar >= 85) + 1
n_90   = np.argmax(cumvar >= 90) + 1

print(f'Components for 85% variance: {n_85}')
print(f'Components for 90% variance: {n_90}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('PCA — Explained Variance', fontsize=12, fontweight='bold')

# Scree plot
ax1.bar(range(1, 21), pca_full.explained_variance_ratio_[:20] * 100, color='steelblue', alpha=0.8)
ax1.set_xlabel('Component'); ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('Scree Plot (first 20 components)')

# Cumulative variance
ax2.plot(range(1, len(cumvar)+1), cumvar, marker='o', ms=3, lw=1.5, color='mediumseagreen')
ax2.axhline(85, color='orange', ls='--', lw=1, label='85%')
ax2.axhline(90, color='crimson', ls='--', lw=1, label='90%')
ax2.axvline(n_85, color='orange', ls=':', lw=1)
ax2.axvline(n_90, color='crimson', ls=':', lw=1)
ax2.set_xlabel('Number of Components'); ax2.set_ylabel('Cumulative Variance (%)')
ax2.set_title('Cumulative Explained Variance')
ax2.set_xlim(1, 42); ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../results/pca_scree.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# Apply PCA retaining 90% variance
N_COMPONENTS = n_90
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X)
print(f'PCA applied: {X.shape[1]} features -> {N_COMPONENTS} components')
print(f'Variance retained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Save PCA-transformed data for downstream use
import pickle
with open('../data/processed/pca_model.pkl', 'wb') as f:
    pickle.dump(pca, f)
np.save('../data/processed/X_pca.npy', X_pca)
print('Saved pca_model.pkl and X_pca.npy')


## 2. K-Means — Elbow + Silhouette Score

We try k=2 to 15 and pick the elbow point — where adding more clusters gives
diminishing returns. We do **not** pick k=2 (normal vs anomaly) since that would
encode label knowledge into the model choice.


In [ ]:
from sklearn.cluster import KMeans as _KM
from sklearn.metrics import silhouette_score

inertias, silhouettes = [], []
K_RANGE = range(2, 16)

print('Computing K-Means for k=2..15 (this may take ~30s)...')
for k in K_RANGE:
    km = _KM(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca)
    inertias.append(km.inertia_)
    # Silhouette on a sample (fast)
    sample_idx = np.random.default_rng(42).choice(len(X_pca), size=min(5000, len(X_pca)), replace=False)
    sil = silhouette_score(X_pca[sample_idx], km.labels_[sample_idx])
    silhouettes.append(sil)
    print(f'  k={k:2d}  inertia={km.inertia_:,.0f}  silhouette={sil:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('K-Means Hyperparameter Selection', fontsize=12, fontweight='bold')

ax1.plot(list(K_RANGE), inertias, marker='o', ms=5, color='steelblue')
ax1.set_xlabel('k (clusters)'); ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')

ax2.plot(list(K_RANGE), silhouettes, marker='o', ms=5, color='mediumseagreen')
ax2.set_xlabel('k (clusters)'); ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score')

plt.tight_layout()
plt.savefig('../results/kmeans_selection.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# Select k at the silhouette peak (unsupervised selection — no labels used)
best_k = list(K_RANGE)[np.argmax(silhouettes)]
print(f'Selected k = {best_k}  (highest silhouette score: {max(silhouettes):.4f})')


In [ ]:
# Fit K-Means anomaly detector with chosen k
km_detector = KMeansAnomalyDetector(k=best_k, random_state=42)
km_detector.fit(X_pca, threshold_percentile=95.0)  # top 5% distances = anomaly

km_scores = km_detector.score(X_pca)
km_preds  = km_detector.predict(X_pca)

print(f'K-Means threshold (95th pct): {km_detector.threshold_:.4f}')
print(f'Flagged as anomaly: {km_preds.sum():,} ({km_preds.mean()*100:.1f}%)')

# Store predictions in labels_df for later use in Notebook 06
labels_df['kmeans_score']   = km_scores
labels_df['kmeans_anomaly'] = km_preds


## 3. K-Means — 2D PCA Scatter Visualisation

Plotting PC1 vs PC2 (first two components) coloured by cluster assignment.
**Note:** the actual model used all `N_COMPONENTS` components — this 2D plot is for
visualisation only; it does not represent the full decision boundary.


In [ ]:
cluster_labels = km_detector.model.labels_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('PCA Scatter (PC1 vs PC2) — K-Means Clusters', fontsize=12, fontweight='bold')

# Left: colour by K-Means cluster
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels,
                          cmap='tab10', s=3, alpha=0.4)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title(f'K-Means Clusters (k={best_k})')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Right: colour by K-Means anomaly flag (NOT ground truth)
axes[1].scatter(X_pca[km_preds==0, 0], X_pca[km_preds==0, 1],
               c='lightsteelblue', s=2, alpha=0.3, label='Normal')
axes[1].scatter(X_pca[km_preds==1, 0], X_pca[km_preds==1, 1],
               c='crimson', s=8, alpha=0.6, label='Flagged anomaly')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('K-Means Anomaly Flags')
axes[1].legend(fontsize=8, markerscale=3)

plt.tight_layout()
plt.savefig('../results/kmeans_scatter.png', dpi=110, bbox_inches='tight')
plt.show()


## 4. DBSCAN — k-Distance Plot for `eps` Selection

The k-distance plot sorts all points by their distance to their k-th nearest neighbour.
The 'knee' in this curve is the recommended `eps` value — it separates dense regions
(normal behaviour) from sparse regions (anomalies).


In [ ]:
K_NN = 5  # typical recommendation: min_samples - 1

print('Computing k-NN distances (this may take ~20s)...')
n_jobs = min(4, os.cpu_count() or 1)
nbrs = NearestNeighbors(n_neighbors=K_NN, n_jobs=n_jobs)
nbrs.fit(X_pca)
distances, _ = nbrs.kneighbors(X_pca)
k_distances  = np.sort(distances[:, K_NN - 1])[::-1]  # distance to k-th neighbour, sorted desc

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(k_distances, lw=1.0, color='steelblue')
ax.set_xlabel('Points (sorted by distance)'); ax.set_ylabel(f'{K_NN}-NN Distance')
ax.set_title(f'k-Distance Plot (k={K_NN}) — choose eps at the knee', fontsize=11, fontweight='bold')

# Add a horizontal line at the chosen eps to show the cut
# Knee: find elbow using max-curvature heuristic
from scipy.signal import savgol_filter
try:
    smooth = savgol_filter(k_distances, window_length=51, polyorder=3)
    abs_diffs = np.abs(np.diff(smooth))
    knee_idx = np.argmax(abs_diffs < abs_diffs.mean() * 0.5)
    eps_auto = float(k_distances[knee_idx])
    max_safe_eps = float(np.percentile(k_distances, 95))
    if eps_auto > max_safe_eps or eps_auto <= 0:
        eps_auto = max_safe_eps
except Exception:
    eps_auto = float(np.percentile(k_distances, 90))

eps_auto = round(eps_auto, 2)
ax.axhline(eps_auto, color='crimson', ls='--', lw=1.5, label=f'Suggested eps = {eps_auto}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../results/dbscan_kdist.png', dpi=110, bbox_inches='tight')
plt.show()
print(f'Suggested eps = {eps_auto}')


In [ ]:
# Fit DBSCAN with selected eps
MIN_SAMPLES = 5
EPS = eps_auto

db_detector = DBSCANAnomalyDetector()
db_preds = db_detector.fit_predict(X_pca, eps=EPS, min_samples=MIN_SAMPLES)
db_scores = db_detector.score(X_pca)

n_clusters = len(set(db_detector.labels_)) - (1 if -1 in db_detector.labels_ else 0)
print(f'DBSCAN: eps={EPS}, min_samples={MIN_SAMPLES}')
print(f'Clusters found   : {n_clusters}')
print(f'Noise points     : {(db_detector.labels_ == -1).sum():,} ({(db_detector.labels_ == -1).mean()*100:.1f}%)')
print(f'Flagged anomaly  : {db_preds.sum():,} ({db_preds.mean()*100:.1f}%)')

labels_df['dbscan_score']   = db_scores
labels_df['dbscan_anomaly'] = db_preds


## 5. DBSCAN — 2D PCA Scatter Visualisation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('PCA Scatter (PC1 vs PC2) — DBSCAN Results', fontsize=12, fontweight='bold')

# Colour by DBSCAN cluster (noise = black)
dbscan_labels = db_detector.labels_
unique_lbls   = sorted(set(dbscan_labels))
colours = plt.cm.tab10(np.linspace(0, 1, max(len(unique_lbls), 2)))

for lbl, col in zip(unique_lbls, colours):
    mask = dbscan_labels == lbl
    name = 'Noise (-1)' if lbl == -1 else f'Cluster {lbl}'
    clr  = 'black'      if lbl == -1 else col
    size = 12            if lbl == -1 else 2
    alpha = 0.7          if lbl == -1 else 0.3
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[clr], s=size,
                   alpha=alpha, label=name if lbl <= 3 else None)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title('DBSCAN Clusters (black = noise/anomaly)')
axes[0].legend(fontsize=7, markerscale=3)

# Anomaly flags
axes[1].scatter(X_pca[db_preds==0, 0], X_pca[db_preds==0, 1],
               c='lightsteelblue', s=2, alpha=0.3, label='Normal')
axes[1].scatter(X_pca[db_preds==1, 0], X_pca[db_preds==1, 1],
               c='crimson', s=10, alpha=0.7, label='Flagged (noise)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('DBSCAN Anomaly Flags')
axes[1].legend(fontsize=8, markerscale=3)

plt.tight_layout()
plt.savefig('../results/dbscan_scatter.png', dpi=110, bbox_inches='tight')
plt.show()


## 6. Save Predictions for Notebook 06


In [ ]:
# Save all predictions + labels to a single CSV for Notebook 06
labels_df.to_csv('../data/processed/predictions_so_far.csv', index=False)
print('Saved: data/processed/predictions_so_far.csv')
print(f'Columns: {labels_df.columns.tolist()}')
print('\nNext: Run 05_anomaly_detection_comparison.ipynb (Isolation Forest)')
